# Quarto Hidden-Information Benchmark V1

Local experiment using the frozen PubChem snapshot and the existing Quarto engine. The benchmark refuses to start a draft or game when pre-draft feasibility is false. The notebook runs trusted local code only; executing an arbitrary Python agent in this kernel is not a secure isolation boundary.

## Current Benchmark Architecture (FMA Hidden-Information Quarto — Series & Automated Evaluator)

1. **Strict Real-Data Instance Validation (`DV-01..07`):** 16 FMA audio tracks with complete 4-bit signature coverage (`0..15`) and strict structural/ID quartet acceptance (`id_level_count > 1` and `structural_count > 1`).
2. **Phase 1 — Alternating Attribute Draft (`enable_draft=True`):** Models observe the 16x18 binary feature matrix (with per-match opaque UUIDv4s) and propose features until each player owns 2 accepted features forming a valid quartet.
3. **Hidden Bit Mapping & Ownership-Aware Win Rule:** Accepted quartet features are mapped via secret bit permutation to `quarto_engine`; players win only by completing a 4-in-a-row line sharing one of **their own 2 drafted attributes**.
4. **Best-Effort 3-Game Counterbalanced Series:** Runs up to 3 games alternating `starting_player` (`player_1`, `player_2`, `player_1`). If budget or API limits prevent subsequent games after at least 1 completed game, completed games are evaluated and reasons for skipped/failed games are recorded without aborting the benchmark.
5. **Privacy-Preserving Automated Evaluator:** Evaluates `restricted_log` inside the trusted boundary and emits aggregated series and per-match metrics (`draft_precision`, `blunder_avoidance_rate`, `self_bit_alignment_rate`, `immediate_win_conversion_rate`) without leaking per-match UUID-to-signature crosswalks.

## Cell A — Dataset loading and scientific data validation

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "quarto_engine.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate quarto_engine.py from the notebook working directory.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hidden_quarto_benchmark import (
    ATTRIBUTE_DEFINITIONS,
    ATTRIBUTE_POOL_VERSION,
    DEFAULT_SEED_SEARCH_LIMIT,
    DEFAULT_PUBLIC_SEED,
    DraftController,
    HiddenQuartoController,
    InfeasibleSetupError,
    OwnershipAwareValidator,
    analyze_pre_draft,
    build_draft_private_observation,
    build_draft_public_observation,
    load_dataset,
    local_experiment_summary,
    permute_piece_id,
    select_elements,
    search_candidate_seeds,
)

dataset = load_dataset()
print({
    "dataset_version": dataset.version,
    "source": dataset.source_url,
    "retrieved_on": dataset.retrieved_on,
    "sha256": dataset.sha256,
    "source_records": dataset.source_record_count,
    "eligible_records": len(dataset.elements),
    "excluded_records": [item.__dict__ if hasattr(item, "__dict__") else (item.atomic_number, item.symbol, item.reason) for item in dataset.excluded_elements],
    "attribute_pool_version": ATTRIBUTE_POOL_VERSION,
    "attribute_count": len(ATTRIBUTE_DEFINITIONS),
})

{'dataset_version': 'pubchem-periodic-table-2026-11-08', 'source': 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/periodictable/JSON', 'retrieved_on': '2026-11-08', 'sha256': 'f2cd308eaa1aef7db2a21f514b3af3ac9a068733042644d2f110ee8982ebd608', 'source_records': 118, 'eligible_records': 117, 'excluded_records': [(113, 'Nh', 'source oxidation-state value is missing')], 'attribute_pool_version': 'chemical-binary-attributes-v1', 'attribute_count': 40}


## Cell B — Deterministic selection of 16 elements

In [2]:
public_seed = DEFAULT_PUBLIC_SEED
elements = select_elements(dataset, public_seed)
selection = local_experiment_summary(dataset, public_seed)
print(selection)

{'dataset_version': 'pubchem-periodic-table-2026-11-08', 'public_seed': 1729, 'selection_algorithm': 'Sort eligible elements by SHA-256(dataset_version|seed|atomic_number); take the first 16, with atomic number as tie-breaker.', 'selected_elements': ['Sn', 'Ge', 'Nd', 'Cr', 'In', 'W', 'Fl', 'Pt', 'Pb', 'Po', 'Mc', 'Ru', 'Ti', 'Ag', 'Ar', 'Mo'], 'combinations_examined': 91390, 'valid_quartet_count': 0, 'mathematically_feasible': False, 'hidden_quartet_ambiguity': 'none'}


## Cell C — Pre-draft feasibility and bounded seed search

The report counts every 4-feature combination. Quartet identities and signatures are not returned by the seed search. The bounded search checks seeds in ascending order and stops at the first sample with at least two valid quartets; it never chooses a quartet or starts a game.

In [3]:
analysis = analyze_pre_draft(elements, dataset.attribute_definitions)
feasibility_report = analysis.report
print({
    "combinations_examined": feasibility_report.combinations_examined,
    "valid_quartet_count": feasibility_report.valid_quartet_count,
    "mathematically_feasible": feasibility_report.mathematically_feasible,
    "hidden_quartet_ambiguity": feasibility_report.hidden_quartet_ambiguity,
})

seed_search = search_candidate_seeds(
    dataset,
    start_seed=public_seed,
    max_candidates=DEFAULT_SEED_SEARCH_LIMIT,
)
first_qualifying = seed_search.first_qualifying_candidate
print({
    "seed_order": f"{seed_search.start_seed} through {seed_search.start_seed + seed_search.max_candidates - 1}",
    "candidate_bound": seed_search.max_candidates,
    "candidates_examined": len(seed_search.candidates),
    "minimum_valid_quartets": seed_search.minimum_valid_quartets,
    "zero_quartet_candidates": seed_search.zero_quartet_candidates,
    "unique_quartet_candidates": seed_search.unique_quartet_candidates,
    "multiple_quartet_candidates": seed_search.multiple_quartet_candidates,
    "first_candidate_results": [
        {
            "seed": candidate.seed,
            "elements": candidate.selected_symbols,
            "valid_quartet_count": candidate.valid_quartet_count,
            "classification": candidate.classification,
        }
        for candidate in seed_search.candidates[:3]
    ],
    "last_candidate_results": [
        {
            "seed": candidate.seed,
            "elements": candidate.selected_symbols,
            "valid_quartet_count": candidate.valid_quartet_count,
            "classification": candidate.classification,
        }
        for candidate in seed_search.candidates[-3:]
    ],
    "first_qualifying_candidate": None if first_qualifying is None else {
        "seed": first_qualifying.seed,
        "elements": first_qualifying.selected_symbols,
        "valid_quartet_count": first_qualifying.valid_quartet_count,
    },
    "game_started": False,
})

{'combinations_examined': 91390, 'valid_quartet_count': 0, 'mathematically_feasible': False, 'hidden_quartet_ambiguity': 'none'}
{'seed_order': '1729 through 2752', 'candidate_bound': 1024, 'candidates_examined': 1024, 'minimum_valid_quartets': 2, 'zero_quartet_candidates': 1024, 'unique_quartet_candidates': 0, 'multiple_quartet_candidates': 0, 'first_candidate_results': [{'seed': 1729, 'elements': ('Sn', 'Ge', 'Nd', 'Cr', 'In', 'W', 'Fl', 'Pt', 'Pb', 'Po', 'Mc', 'Ru', 'Ti', 'Ag', 'Ar', 'Mo'), 'valid_quartet_count': 0, 'classification': 'zero'}, {'seed': 1730, 'elements': ('Bh', 'Ag', 'Cu', 'Cd', 'Ce', 'Cs', 'Np', 'Yb', 'Md', 'Ac', 'Hg', 'Db', 'F', 'Ir', 'Cr', 'Ru'), 'valid_quartet_count': 0, 'classification': 'zero'}, {'seed': 1731, 'elements': ('Sg', 'Li', 'Og', 'Ca', 'Bi', 'Rn', 'Cr', 'At', 'Bk', 'As', 'Tb', 'Rb', 'Te', 'Ag', 'Y', 'Lu'), 'valid_quartet_count': 0, 'classification': 'zero'}], 'last_candidate_results': [{'seed': 2750, 'elements': ('Pa', 'Au', 'Sm', 'In', 'Fe', 'Fm', 'N

## Cell D — Ownership-aware incremental validator

A rejected proposal changes neither accepted attributes nor the proposer. Feedback is always exactly `accepted` or `rejected` and is visible only in that proposer's private observation.

In [4]:
draft = None
if feasibility_report.mathematically_feasible:
    validator = OwnershipAwareValidator(analysis)
    draft = DraftController(validator)
    print("Setup is feasible; submit proposals through DraftController.submit_proposal.")

## Cell E — Private attribute-to-engine-bit mapping

The mapping is generated only after a feasible draft completes. It uses system randomness independent of the public seed and is retained only inside the trusted controller.

In [5]:
secret_mapping = None
if feasibility_report.mathematically_feasible and draft is not None and draft.is_complete:
    controller = HiddenQuartoController(
        dataset,
        elements,
        public_seed,
        draft.completed_attributes(),
        private_feedback={
            "player_1": draft.private_feedback_for("player_1"),
            "player_2": draft.private_feedback_for("player_2"),
        },
    )
    print("Private mapping created inside controller; mapping values are not displayed.")
else:
    print("Mapping not created: the current setup/draft is not eligible for gameplay.")

Mapping not created: the current setup/draft is not eligible for gameplay.


## Cell F — Hidden-information controller and observation builders

Public draft observations contain the element data, definitions, current phase/player, and legal proposals. Private draft observations add only the recipient's own accepted attributes and feedback. Controller objects themselves are trusted and must not be handed to arbitrary same-process Python agents.

In [6]:
if draft is None:
    print("Draft observations skipped because pre-draft feasibility failed.")
else:
    public_draft_observation = build_draft_public_observation(
        dataset, elements, public_seed, draft
    )
    player_1_observation = build_draft_private_observation(
        dataset, elements, public_seed, draft, "player_1"
    )
    player_2_observation = build_draft_private_observation(
        dataset, elements, public_seed, draft, "player_2"
    )
    assert "private" not in public_draft_observation
    assert player_1_observation["private"]["proposal_feedback"] in (None, "accepted", "rejected")
    assert player_2_observation["private"]["proposal_feedback"] in (None, "accepted", "rejected")
    print({
        "public_keys": sorted(public_draft_observation),
        "player_1_private": player_1_observation["private"],
        "player_2_private": player_2_observation["private"],
    })

Draft observations skipped because pre-draft feasibility failed.


## Cell G — Existing Quarto engine integration

The controller translates public element identifiers to internal piece IDs before calling `select_piece` and `place_piece`; it translates engine history and terminal results back to sanitized element identities/cell indices. The following engine-level check verifies bit-permutation invariance without inventing element property data.

In [7]:
from itertools import permutations
from quarto_engine import find_winning_lines

for permutation in permutations(range(4)):
    board = [None] * 16
    for cell, piece_id in enumerate((0, 2, 4, 6)):
        board[cell] = permute_piece_id(piece_id, permutation)
    wins = find_winning_lines(board)
    assert len(wins) == 1 and wins[0].cells == (0, 1, 2, 3)
print("All 24 bit permutations preserve the standard Quarto win rule.")

All 24 bit permutations preserve the standard Quarto win rule.


## Cell H — Automated tests and first local experiment

This cell runs the project regression suite and the pre-draft experiment. If the chosen seed is infeasible, the result is reported and no hidden-information game is started.

In [8]:
import subprocess

test_run = subprocess.run(
    [sys.executable, "-B", "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=False,
)
print(test_run.stdout)
print(test_run.stderr)
if test_run.returncode != 0:
    raise RuntimeError(f"Test suite failed with exit code {test_run.returncode}.")

from quarto_runner import run_game
baseline = run_game("classic", seed=public_seed)
assert baseline.state.is_terminal
print({
    "baseline": "existing classic RandomAgent game completed",
    "actions": baseline.action_count,
    "benchmark_setup_feasible": feasibility_report.mathematically_feasible,
    "benchmark_status": (
        "ready for draft/game"
        if feasibility_report.mathematically_feasible
        else "blocked: zero valid quartets for the selected seed"
    ),
})


test_attribute_values_are_complete_for_selected_public_elements (test_hidden_quarto_benchmark.HiddenQuartoBenchmarkTests.test_attribute_values_are_complete_for_selected_public_elements) ... ok
test_bit_permutation_preserves_quarto_win_rule_for_all_permutations (test_hidden_quarto_benchmark.HiddenQuartoBenchmarkTests.test_bit_permutation_preserves_quarto_win_rule_for_all_permutations) ... ok
test_bounded_seed_search_reports_exact_counts_and_public_samples (test_hidden_quarto_benchmark.HiddenQuartoBenchmarkTests.test_bounded_seed_search_reports_exact_counts_and_public_samples) ... ok
test_frozen_dataset_and_binary_attribute_schema (test_hidden_quarto_benchmark.HiddenQuartoBenchmarkTests.test_frozen_dataset_and_binary_attribute_schema) ... ok
test_infeasible_analysis_blocks_draft_before_any_proposal (test_hidden_quarto_benchmark.HiddenQuartoBenchmarkTests.test_infeasible_analysis_blocks_draft_before_any_proposal) ... ok
test_infeasible_seed_blocks_gameplay_explicitly (test_hidden_quarto_